# Medicare Part D Veri Seti Filtreleme ve Birleştirme

Bu notebook, Medicare Part D ham verilerini filtreler ve birleştirir.

## Not:
- Ham Medicare verileri çok büyük olduğundan (~GB), bu notebook sadece filtreleme amacıyla kullanılır.
- İşlenmiş veri seti zaten `../data/` klasöründe mevcut:
  - `Combined_LEIE_Medicare_2017_2019_DOWNSIZED_1mil.csv`

## Adımlar:
1. Seçili sütunları filtrele ve kaydet
2. Farklı yılları birleştir

## 1. Veri Setlerini Seçili Sütunlara Göre Filtrele ve Kaydet

In [4]:
import pandas as pd
import gc
import os

# Data klasörünü oluştur
os.makedirs('../data', exist_ok=True)

# Saklanacak sütunlar
use_cols = ['Prscrbr_NPI', 'Prscrbr_Type', 'Tot_Benes', 'Tot_Clms',
            'Tot_30day_Fills', 'Tot_Day_Suply', 'Tot_Drug_Cst']

# Bellek tasarrufu için veri tipleri
dtype_map = {
    'Prscrbr_NPI': 'str',
    'Prscrbr_Type': 'category',
    'Tot_Benes': 'float32',
    'Tot_Clms': 'float32',
    'Tot_30day_Fills': 'float32',
    'Tot_Day_Suply': 'float32',
    'Tot_Drug_Cst': 'float32',
}

# https://data.cms.gov/provider-summary-by-type-of-service/medicare-part-d-prescribers/medicare-part-d-prescribers-by-provider-and-drug/data/2017
# https://data.cms.gov/provider-summary-by-type-of-service/medicare-part-d-prescribers/medicare-part-d-prescribers-by-provider-and-drug/data/2018
# https://data.cms.gov/provider-summary-by-type-of-service/medicare-part-d-prescribers/medicare-part-d-prescribers-by-provider-and-drug/data/2019
# 2017-2019 yılları için döngü
for year in range(2017, 2020):
    file_name = f"../data/Medicare_Part_D_Prescribers_by_Provider_and_Drug_{year}.csv"
    output_name = f"../data/Filtered_Medicare_{year}.csv"

    try:
        print(f"İşleniyor: {file_name}")
        df = pd.read_csv(file_name, usecols=use_cols, dtype=dtype_map)
        df.to_csv(output_name, index=False)
        print(f"✓ Kaydedildi: {output_name} ({len(df)} satır)")
        del df
        gc.collect()
    except FileNotFoundError:
        print(f"⚠ Dosya bulunamadı: {file_name}")
        print("  → Ham Medicare verileri gerekli. CMS web sitesinden indirin.")
    except Exception as e:
        print(f"✗ Hata: {file_name}: {e}")

İşleniyor: ../data/Medicare_Part_D_Prescribers_by_Provider_and_Drug_2017.csv
✓ Kaydedildi: ../data/Filtered_Medicare_2017.csv (25209130 satır)
İşleniyor: ../data/Medicare_Part_D_Prescribers_by_Provider_and_Drug_2018.csv
✓ Kaydedildi: ../data/Filtered_Medicare_2018.csv (25311600 satır)
İşleniyor: ../data/Medicare_Part_D_Prescribers_by_Provider_and_Drug_2019.csv
✓ Kaydedildi: ../data/Filtered_Medicare_2019.csv (25401870 satır)


## 2. Farklı Yılları Birleştir

In [ ]:
import pandas as pd
import gc

def combine_and_save(years, output_filename):
    """
    Filtrelenmiş CSV dosyalarını birleştirir ve kaydeder.
    
    Args:
        years: Birleştirilecek yılların listesi
        output_filename: Çıktı dosyası adı
    """
    dfs = []
    for year in years:
        file_name = f"../data/Filtered_Medicare_{year}.csv"
        try:
            print(f"Yükleniyor: {file_name}")
            df = pd.read_csv(file_name)
            df['Source_Year'] = year  # Yıl bilgisi ekle
            dfs.append(df)
            print(f"  → {len(df)} satır yüklendi")
        except FileNotFoundError:
            print(f"⚠ Dosya bulunamadı: {file_name}")
        except Exception as e:
            print(f"✗ Hata: {file_name}: {e}")
    
    # Birleştir ve kaydet
    if dfs:
        combined_df = pd.concat(dfs, ignore_index=True)
        combined_df.to_csv(output_filename, index=False)
        print(f"\n✓ Birleştirilmiş dosya kaydedildi: {output_filename}")
        print(f"  → Toplam {len(combined_df)} satır")
        del combined_df, dfs
        gc.collect()
    else:
        print(f"⚠ Kaydedilecek veri yok: {output_filename}")

# 2017-2019 verilerini birleştir (data klasörüne kaydet)
combine_and_save([2017, 2018, 2019], "../data/Combined_Medicare_2017_2019.csv")

Yükleniyor: ../data/Filtered_Medicare_2017.csv
  → 25209130 satır yüklendi
Yükleniyor: ../data/Filtered_Medicare_2018.csv
  → 25311600 satır yüklendi
Yükleniyor: ../data/Filtered_Medicare_2019.csv
  → 25401870 satır yüklendi


## 3. Mevcut İşlenmiş Veriyi Kontrol Et

Eğer yukarıdaki adımları atlarsanız, hazır veri setini kullanabilirsiniz:

In [ ]:
import pandas as pd
import os

# İşlenmiş veri seti konumu
processed_file = "../data/Combined_LEIE_Medicare_2017_2019_DOWNSIZED_1mil.csv"

if os.path.exists(processed_file):
    print(f"✓ İşlenmiş veri seti mevcut: {processed_file}")
    
    # İlk 5 satırı kontrol et
    df = pd.read_csv(processed_file, nrows=5)
    print(f"\nSütunlar: {df.columns.tolist()}")
    print(f"\nÖrnek veri:")
    print(df.head())
else:
    print(f"⚠ İşlenmiş veri bulunamadı: {processed_file}")
    print("  → Lütfen LEIE merging notebook'unu çalıştırın.")